# CPSC 425 — Assignment 2
## Scaled Representations, Face Detection, and Image Blending

**Due:** 11:59 pm, Tuesday, February 10, 2026  
**Total:** 45 points

This assignment explores normalized cross-correlation (NCC) for object detection in a scaled representation and pyramid-based image blending. You will implement Gaussian and Laplacian pyramids, detect faces at multiple scales, evaluate detection quality, and blend image pairs with a Gaussian mask.

Complete every `TODO` cell, preserve the required function names and signatures, and add clear comments where your implementation is not self-evident. Run all test and visualization cells before submitting. If the data files are not beside this notebook, update `DATA_DIR` below.

Submit this notebook and a PDF export. The PDF should include the required pyramid visualizations, detections for all six images, threshold and recall analysis, reconstructed pyramids, mask pyramid, and blend results.

In [ ]:
from PIL import Image, ImageDraw
import numpy as np
import math
from scipy import signal
from scipy.ndimage import gaussian_filter
from ncc import normxcorr2D

from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR = Path('.')
SCALE = 0.75
MIN_SIZE = 64
TEMPLATE_WIDTH = 15

def open_image(name, mode=None):
    """Open an assignment image from DATA_DIR, optionally converting its mode."""
    image = Image.open(DATA_DIR / name)
    return image.convert(mode) if mode else image

def t2i(t):
    return Image.fromarray(np.clip(t, 0, 255).astype(np.uint8))

def i2t(i):
    return np.asarray(i).astype(np.float32)


## Part 1 — Face detection (25 points)

The supplied archive contains `ncc.py`, the face template, and six test images: `judybats`, `students`, `tree`, `family`, `fans`, and `sports`. The template should be resized to `TEMPLATE_WIDTH` pixels wide while preserving its aspect ratio. `ncc.normxcorr2D(image, template)` returns an NCC array with the same spatial size as `image`.

### 1. Gaussian pyramid (2 points)

Return the original image followed by reduced images at scale `scale`. Filter each channel before downsampling with a Gaussian whose sigma is `1/(2*scale)`. Stop before the next reduction would make the larger spatial dimension smaller than `minsize`. A list of PIL images or a list of NumPy arrays is acceptable, but use one representation consistently.

In [ ]:
def MakeGaussianPyramid(image: Image, scale: float, minsize: int):
    """Build a Gaussian pyramid, stopping before the next level is too small."""
    list_out = [i2t(image)]
    while True:
        height, width = list_out[-1].shape[:2]
        next_width = int(width * scale)
        next_height = int(height * scale)
        if max(next_width, next_height) < minsize:
            break

        filtered = gaussian_filter(
            np.asarray(list_out[-1]), 1 / (2 * scale), axes=(0, 1)
        )
        new_image = t2i(filtered).resize(
            (next_width, next_height), Image.BICUBIC
        )
        list_out.append(i2t(new_image))

    return list_out


def ShowGaussianPyramid(pyramid):
    """Join pyramid levels horizontally and display the result."""
    # TODO: support the representation chosen in MakeGaussianPyramid.
    offset = 0
    widths, heights = zip(*((i.shape[1], i.shape[0]) for i in pyramid))

    total_width = sum(widths)
    max_height = max(heights)

    new_im = Image.new("RGB", (total_width, max_height))

    for tensor in pyramid:
        new_im.paste(t2i(tensor), (offset, 0))
        offset += tensor.shape[1]

    display(new_im)



### 2. Check the Gaussian pyramid (3 points)

Use this cell to test your pyramid on a grayscale image and include the displayed result in the PDF. Also verify that your implementation handles an RGB image, since Part 2 requires color support.

In [ ]:
# TODO: replace filenames with the names in your downloaded archive.
gray_test = open_image('judybats.jpg', 'L')
gray_pyramid = MakeGaussianPyramid(gray_test, SCALE, MIN_SIZE)
ShowGaussianPyramid(gray_pyramid)
print('levels:', len(gray_pyramid))

rgb_test = open_image('students.jpg', 'RGB')
rgb_pyramid = MakeGaussianPyramid(rgb_test, SCALE, MIN_SIZE)
ShowGaussianPyramid(rgb_pyramid)


### 3. Template matching (10 points)

Implement `FindTemplate`. Resize the template to `TEMPLATE_WIDTH`, compute NCC at each pyramid level, and draw a rectangle for every coefficient above `threshold`. All rectangles must be drawn on a copy of the original image, with coordinates scaled back by the appropriate power of `scale`. The result should be RGB so the rectangles are visible.

In [ ]:
FACE_TEMPLATE_ORIGINAL = open_image("face_detection_template.jpg")
FACE_TEMPLATE_ORIGINAL_W, FACE_TEMPLATE_ORIGINAL_H = FACE_TEMPLATE_ORIGINAL.width, FACE_TEMPLATE_ORIGINAL.height
TEMPLATE_HEIGHT = int(FACE_TEMPLATE_ORIGINAL_H*TEMPLATE_WIDTH/FACE_TEMPLATE_ORIGINAL_W)
FACE_TEMPLATE = FACE_TEMPLATE_ORIGINAL.resize((TEMPLATE_WIDTH, TEMPLATE_HEIGHT))
# display(FACE_TEMPLATE)

def FindTemplate(pyramid, template, threshold):
    """Mark every pyramid location whose NCC exceeds threshold.

    TODO: return an RGB PIL image containing all detections overlaid on the
    original-resolution image. Account for the level scale and template size.
    """
    output = Image.fromarray(pyramid[0].astype(np.uint8)).convert("RGB").copy()
    draw = ImageDraw.Draw(output)

    template_width, template_height = template.size

    for level, tensor in enumerate(pyramid):
        corr = normxcorr2D(t2i(tensor).convert("L"), template)
        detections = np.argwhere(corr > threshold)
        scale = SCALE ** level

        for row, col in detections:
            # np.argwhere gives row, col = y, x
            x = int(col / scale)
            y = int(row / scale)

            width = int(template_width / scale)
            height = int(template_height / scale)

            draw.rectangle(
                (x - width/2, y- height/2, x + width/2, y + height/2),
                outline="red",
                width=2
            )

    return output



# TODO: load the template and one test image, then inspect the detections.
display(FindTemplate(gray_pyramid, FACE_TEMPLATE, 0.7))


### 4. Equal-error threshold (5 points)

Using the six images (`judybats`, `students`, `tree`, `family`, `fans`, `sports`), choose one threshold for the complete set—not one threshold per image. Count false positives and false negatives and select a threshold close to an equal error rate. Document the threshold and show detections for all six images.

In [ ]:
IMAGE_NAMES = ['judybats.jpg', 'students.jpg', 'tree.jpg',
               'family.jpg', 'fans.jpg', 'sports.jpg']

# TODO: define how you count false positives and false negatives for the
# provided images, evaluate THRESHOLDS, and record your selected threshold.
# The ground-truth face counts may be recorded here if they are supplied by
# the archive or course staff.




selected_threshold = 0.60
print(f"chosen threshold: {selected_threshold}")

# Following code generated with chatgpt
results = []

for name in IMAGE_NAMES:
    image = open_image(name, "RGB")
    pyramid = MakeGaussianPyramid(image, SCALE, MIN_SIZE)
    result = FindTemplate(pyramid, FACE_TEMPLATE, selected_threshold)
    results.append((name, result))

fig, axes = plt.subplots(
    1,
    len(results),
    figsize=(24, 6),
    squeeze=False
)

for ax, (name, result) in zip(axes[0], results):
    ax.imshow(result)
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()


### 5. Recall analysis (5 points)

For the threshold chosen above, report the recall rate for each image. Recall is `true positives / (true positives + false negatives)`. Explain why NCC can have very low recall on some images (for example, changes in scale, pose, illumination, occlusion, or image content).

### ANSWER:

I define a "true postive" to be any number of correctly scaled correlations on a human face

a "false positive" is an activation on anything that is a not a human face, or a highly incorrect scaling on a human face(ie: a nose or other subfeature)

a "false negative" is a when activation does not occur on the defined true positive activation class.

#### RESULTS

judybats: TP=4, FP=1, FN=1, Recall=80.0%, Error ratio=1.00
students: TP=23, FP=2, FN=4, Recall=85.2%, Error ratio=0.50
tree: TP=0, FP=2, FN=0, Recall=N/A, Error ratio=∞
family: TP=2, FP=0, FN=1, Recall=66.7%, Error ratio=0.00
fans: TP=0, FP=4, FN=3, Recall=0.0%, Error ratio=1.33
sports: TP=0, FP=1, FN=1, Recall=0.0%, Error ratio=1.00
Total: TP=29, FP=10, FN=10, Recall=74.4%, Error ratio=1.00

**Why NCC has a low recall rate on some images:**  

**Threshold selection / equal error rate:**  
I chose a single threshold of `0.6` for all six images. At this threshold, the totals are `29` true positives, `10` false positives, and `10` false negatives. The overall recall is `74.4%`, and the false-positive-to-false-negative ratio is `1.00`.

**Why NCC has a low recall rate on some images:**  

The filter is of a white face, so we can expect a lack of activations for people with significantly different skin tones. I also considered excluding faces with closed eyes but figured it was not in the spirit of the question.

## Part 2 — Image blending (15 points)

Pyramid blending combines Laplacian pyramids of two images with a Gaussian pyramid mask. Use RGB arrays with three channels. The Gaussian and Laplacian pyramids must have the same number of levels, and the highest-resolution level must match the input dimensions.

In [ ]:
def MakeLaplacianPyramid(image, scale, minsize):
    """Build an RGB Laplacian pyramid as a list of float NumPy arrays."""
    # TODO: construct a Gaussian pyramid, subtract expanded next levels,
    # and keep the smallest Gaussian level as the final Laplacian level.
    gaussian_pyramid = MakeGaussianPyramid(image, scale, minsize)
    laplacian_pyramid = []
    for idx, tensor_gaussian in enumerate(gaussian_pyramid[:-1]):
        h, w = tensor_gaussian.shape[0], tensor_gaussian.shape[1] 
        tensor_next_level = i2t(t2i(gaussian_pyramid[idx+1]).resize((w, h), Image.BICUBIC))
        laplacian_pyramid.append(tensor_gaussian-tensor_next_level)
    laplacian_pyramid.append(np.asarray(gaussian_pyramid[-1]))
    return laplacian_pyramid


def ShowLaplacianPyramid(pyramid):
    """Display a horizontal visualization of an RGB Laplacian pyramid."""
    # TODO: offset/clip all but the final level for display only.
    # print(pyramid[-1])
    ShowGaussianPyramid(list(map(lambda x: x+128, pyramid[:-1])) + [pyramid[-1]])


def ReconstructGaussianFromLaplacianPyramid(lPyramid):
    """Reconstruct and return the Gaussian pyramid from a Laplacian pyramid."""
    pyramid = list(reversed(lPyramid))

    # The smallest Gaussian level is the final Laplacian level.
    pyramid_out = [i2t(t2i(pyramid[0]))]

    # Reconstruct from the smallest level toward the original resolution.
    for laplacian_level in pyramid[1:]:
        height, width = laplacian_level.shape[:2]

        expanded = t2i(pyramid_out[-1]).resize(
            (width, height),
            Image.Resampling.BICUBIC
        )
        expanded = i2t(expanded)

        reconstructed = expanded + laplacian_level
        pyramid_out.append(reconstructed)

    return list(reversed(pyramid_out))


        


### 1. Laplacian pyramids and reconstruction (3 + 2 + 5 points)

Visualize the Laplacian pyramids for `orchid.jpg` and `violet.jpg`. Reconstruct Gaussian pyramids from each Laplacian pyramid and visualize both reconstructions with `ShowGaussianPyramid`. The final level of the Laplacian pyramid is the smallest Gaussian level; do not apply the display offset when computing the pyramid.

In [ ]:
# TODO: run the required orchid/violet experiments.
orchid = open_image('orchid.jpg', 'RGB')
violet = open_image('violet.jpg', 'RGB')
orchid_lp = MakeLaplacianPyramid(orchid, SCALE, MIN_SIZE)
violet_lp = MakeLaplacianPyramid(violet, SCALE, MIN_SIZE)
ShowLaplacianPyramid(orchid_lp)
ShowGaussianPyramid(ReconstructGaussianFromLaplacianPyramid(orchid_lp))
ShowLaplacianPyramid(violet_lp)
ShowGaussianPyramid(ReconstructGaussianFromLaplacianPyramid(violet_lp))


### 2. Mask pyramid and blending (2 + 5 points)

Build and display a Gaussian pyramid for `orchid_mask.bmp`. Blend the Laplacian pyramids level by level:

`comp[i] = lapA[i] * mask[i] + lapB[i] * (1 - mask[i])`

Normalize the mask to `[0, 1]` before applying it, and display the highest-resolution reconstructed level. The output should have the same dimensions as both input images.

In [ ]:
def BlendLaplacianPyramids(lap_a, lap_b, mask_pyramid):
    """Blend grayscale or RGB Laplacian pyramid levels using grayscale masks."""

    out = []

    for a, b, mask in zip(lap_a, lap_b, mask_pyramid):
        alpha = mask.astype(float) / 255.0  # for masks with values in [0, 255]

        if a.ndim == 3:  # RGB image level: (height, width, channels)
            alpha = alpha[..., None]

        out.append(a * alpha + b * (1.0 - alpha))

    return out

# TODO: construct the mask pyramid, blend orchid/violet, and display output.
mask = open_image('orchid_mask.bmp', 'L')
mask_pyramid = MakeGaussianPyramid(mask, SCALE, MIN_SIZE)
ShowGaussianPyramid(mask_pyramid)
blended_lp = BlendLaplacianPyramids(orchid_lp, violet_lp, mask_pyramid)
blended = ReconstructGaussianFromLaplacianPyramid(blended_lp)[0]
display(t2i(blended))


### 3. Additional experiments (3 points)

Repeat the blend for `blue_cup.jpg` and `green_cup.jpg`, and for `apple.jpg` and `tomato.jpg`. You may also create additional blends. Include the two required outputs below and label every figure clearly.

In [ ]:
# TODO: run and display the blue-cup/green-cup and apple/tomato blends.
def blend_pair(name_a, name_b, mask_name):
    image_a = open_image(name_a, 'RGB')
    image_b = open_image(name_b, 'RGB')
    mask = open_image(mask_name, 'L')
    lp_a = MakeLaplacianPyramid(image_a, SCALE, MIN_SIZE)
    lp_b = MakeLaplacianPyramid(image_b, SCALE, MIN_SIZE)
    mp = MakeGaussianPyramid(mask, SCALE, MIN_SIZE)
    return ReconstructGaussianFromLaplacianPyramid(BlendLaplacianPyramids(lp_a, lp_b, mp))[0]
display(t2i(blend_pair('blue_cup.jpg', 'green_cup.jpg', 'cup_mask.bmp')))
display(t2i(blend_pair('apple.jpg', 'tomato.jpg', 'tomato_mask.bmp')))


In [ ]:
!jupyter-nbconvert hw2.ipynb --to pdf

## Submission checklist

- [ ] All required functions are implemented and documented.
- [ ] Gaussian pyramid test output is included.
- [ ] One global face-detection threshold is reported.
- [ ] Face detections for all six images are included.
- [ ] Recall is reported for each image with an explanation of low-recall cases.
- [ ] Laplacian pyramids for orchid/orchard and violet are included.
- [ ] Both reconstructed Gaussian pyramids and the mask pyramid are included.
- [ ] Orchid/violet, blue-cup/green-cup, and apple/tomato blend outputs are included.
- [ ] The notebook has been run from top to bottom and exported to PDF.